# 18. 최종 비교 — 미래 정보 완전 제거 + 합산(사전 고정) + 차이 신뢰구간 + 0 모드 제거 재계산

**미래 정보 규칙**: 모든 점수는 그 샷 이전 정보 또는 labeled와 겹치지 않는 reference(9~10월 unlabeled, **0 모드 제거**)만 사용. `T_global` 제거.
합산 점수도 평가 파일 안 순위가 아니라 **reference 분포 대비 백분위**로 변환 → 다른 테스트 샷을 보지 않음

**방법 (결과 보기 전에 고정)**
- 팀원: `T_prev10`, `T_past10`, `T_phys300`
- 나: `M_knn_free`, `M_maha_free`, `M_rule_phys`
- 합산: `C_maha+phys300`, `C_maha+past10` = 두 점수의 reference 대비 이상도(−log10(1−pct)) 평균

**A** 두 틀 비교 → **B** 두 방법 차이 부트스트랩 → **C** 0 모드 제거 reference로 13·15번 재계산

In [1]:
%load_ext autoreload
%autoreload 2
import os, json
import numpy as np, pandas as pd
from compare_frames import (load_all_shots, load_unlabeled_normal, official_rows, t_prev, build_ref_nozero,
                            my_scores, frame1_eval, frame2_oof, frame2_eval)
from final_compare import teammate_scores_with_ref, combine, paired_bootstrap
from model_phys import add_phys
from scope import apply_scope
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
OUT = './outputs/final_compare'
os.makedirs(OUT, exist_ok=True)
PRODUCTS = ['cn7', 'rg3']
SCALES = json.load(open('./base/reference_scales.json', encoding='utf-8'))
REF_PERIOD = ('2020-09-01', '2020-11-30')
REF_CACHE = './base/ref_{}_features_nozero_' + f'{REF_PERIOD[0]}_{REF_PERIOD[1]}.csv'   # 17번에서 만든 캐시

## A-1. 점수 계산

In [2]:
SC, REFSC, FEAT, REFNZ = {}, {}, {}, {}
for p in PRODUCTS:
    sh = load_all_shots(p)
    unl = load_unlabeled_normal(p)
    f = add_phys(pd.read_csv(f'./base/{p}_features.csv', parse_dates=['TimeStamp']))
    f['Reason'] = f['Reason'].fillna('정상')
    cache = REF_CACHE.format(p)
    ref = add_phys(pd.read_csv(cache, parse_dates=['TimeStamp'])) if os.path.exists(cache) else build_ref_nozero(p, REF_PERIOD, SCALES)
    FEAT[p], REFNZ[p] = f, ref

    S, R = {}, {}
    S['T_prev10'] = pd.Series(t_prev(sh, 10), index=sh['TimeStamp'].values)          # reference 분포 없음 (합산엔 미사용)
    for k, (a, b) in teammate_scores_with_ref(sh, unl, ref).items():
        S[k] = pd.Series(a, index=sh['TimeStamp'].values); R[k] = b
    for k, (a, b) in my_scores(f, ref).items():
        S[k] = pd.Series(a, index=f['TimeStamp'].values); R[k] = b
    # 합산 (사전 고정) — 같은 샷 시각 기준으로 정렬
    ts = f['TimeStamp'].values
    for name, (m1, m2) in {'C_maha+phys300': ('M_maha_free', 'T_phys300'),
                           'C_maha+past10': ('M_maha_free', 'T_past10')}.items():
        a1, a2 = S[m1].reindex(ts).fillna(0).values, S[m2].reindex(ts).fillna(0).values
        s, r = combine([(a1, R[m1]), (a2, R[m2])])
        S[name] = pd.Series(s, index=ts)
    SC[p], REFSC[p] = S, R
    print(p, '완료:', list(S))

[cn7] 비라벨 28744샷 중 0 모드 18139 (63.1%) 제거


c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


cn7 완료: ['T_prev10', 'T_past10', 'T_phys300', 'M_knn_free', 'M_maha_free', 'M_rule_phys', 'C_maha+phys300', 'C_maha+past10']
[rg3] 비라벨 29633샷 중 0 모드 13119 (44.3%) 제거


c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = geta

rg3 완료: ['T_prev10', 'T_past10', 'T_phys300', 'M_knn_free', 'M_maha_free', 'M_rule_phys', 'C_maha+phys300', 'C_maha+past10']


## A-2. ① 팀원 틀 (공식 2차, 행 단위)

In [3]:
F1T = pd.concat([frame1_eval(p, official_rows(p), SC[p]) for p in PRODUCTS], ignore_index=True)
F1T = F1T.sort_values(['제품', 'AP'], ascending=[True, False])
F1T.to_csv(f'{OUT}/frame1.csv', index=False, encoding='utf-8-sig')
display(F1T.drop(columns=['무작위 상단']).round(3))

,제품,방법,행,불량 행,AP,무작위 평균 대비,유의,상위1%,상위5%,상위10%,F1@5%,TP@5%,F1@10%,TP@10%,정보 상한 F1,F1@5% / 상한
4,cn7,M_maha_free,1211,17,0.373,19.655,O,6/17,6/17,8/17,0.154,6,0.116,8,0.756,0.204
3,cn7,M_knn_free,1211,17,0.346,18.227,O,6/17,7/17,7/17,0.179,7,0.101,7,0.756,0.238
7,cn7,C_maha+past10,1211,17,0.303,15.984,O,6/17,8/17,8/17,0.205,8,0.116,8,0.756,0.271
1,cn7,T_past10,1211,17,0.205,10.793,O,2/17,7/17,10/17,0.179,7,0.145,10,0.756,0.238
6,cn7,C_maha+phys300,1211,17,0.194,10.223,O,6/17,7/17,7/17,0.179,7,0.101,7,0.756,0.238
5,cn7,M_rule_phys,1211,17,0.161,8.470,O,6/17,7/17,8/17,0.179,7,0.116,8,0.756,0.238
2,cn7,T_phys300,1211,17,0.150,7.884,O,4/17,7/17,7/17,0.179,7,0.101,7,0.756,0.238
0,cn7,T_prev10,1211,17,0.125,6.594,O,2/17,5/17,10/17,0.154,6,0.145,10,0.756,0.204
13,rg3,M_rule_phys,1182,25,0.034,1.251,-,1/25,1/25,3/25,0.024,1,0.042,3,0.667,0.036
11,rg3,M_knn_free,1182,25,0.024,0.904,-,0/25,1/25,2/25,0.024,1,0.028,2,0.667,0.036


## A-3. ② 내 틀 (labeled 전체, 샷 단위, 초기허용 제외)

In [4]:
O2 = []
for p in PRODUCTS:
    f = apply_scope(FEAT[p][FEAT[p]['split'] != '제외'], exclude_initial=True)
    O2.append(frame2_oof(p, f, SC[p]))          # 순위 지표만 쓰므로 원점수 그대로
O2 = pd.concat(O2, ignore_index=True)
F2T = frame2_eval(O2)
F2T.to_csv(f'{OUT}/frame2.csv', index=False, encoding='utf-8-sig')
cols = ['product', 'method', 'AP', 'AP_lo', 'AP_hi', 'AP/base', 'hit@10%', '전체 상위10%', 'F1@5%(전체)',
        'AP_가스', 'hit10_가스', 'AP_미성형', 'hit10_미성형']
display(F2T[cols].sort_values(['product', 'AP'], ascending=[True, False]).round(3))

,product,method,AP,AP_lo,AP_hi,AP/base,hit@10%,전체 상위10%,F1@5%(전체),AP_가스,hit10_가스,AP_미성형,hit10_미성형
0,cn7,M_maha_free,0.212,0.011,0.416,27.843,4/15,5/15,0.070,0.007,1/11,0.751,3/4
1,cn7,M_knn_free,0.129,0.010,0.354,17.027,4/15,4/15,0.070,0.007,1/11,0.442,3/4
2,cn7,C_maha+past10,0.117,0.016,0.342,15.394,5/15,5/15,0.088,0.011,2/11,0.376,3/4
3,cn7,C_maha+phys300,0.092,0.013,0.329,12.049,5/15,5/15,0.070,0.009,2/11,0.288,3/4
4,cn7,M_rule_phys,0.090,0.011,0.310,11.859,5/15,5/15,0.070,0.008,2/11,0.288,3/4
5,cn7,T_phys300,0.073,0.013,0.265,9.667,7/15,5/15,0.070,0.009,4/11,0.228,3/4
6,cn7,T_past10,0.038,0.014,0.152,5.065,6/15,6/15,0.070,0.014,3/11,0.065,3/4
7,cn7,T_prev10,0.035,0.014,0.125,4.606,5/15,6/15,0.053,0.013,2/11,0.056,3/4
8,rg3,M_rule_phys,0.085,0.057,0.181,1.660,5/32,5/32,0.063,0.072,4/22,0.027,1/10
9,rg3,T_phys300,0.080,0.051,0.148,1.570,2/32,7/32,0.159,0.059,1/22,0.034,1/10


## B. 두 방법 차이 — 부트스트랩 95% CI (①은 같은 샷의 LH/RH를 함께 뽑음)

In [5]:
PAIRS = [('C_maha+phys300', 'M_maha_free'), ('C_maha+phys300', 'T_phys300'),
         ('M_maha_free', 'T_past10'), ('M_maha_free', 'T_phys300'), ('C_maha+past10', 'T_past10')]
BOOT = []
for p in PRODUCTS:
    rows = official_rows(p)
    f = apply_scope(FEAT[p][FEAT[p]['split'] != '제외'], exclude_initial=True)
    for a, b in PAIRS:
        # ① 행 단위
        sa, sb = rows['TimeStamp'].map(SC[p][a]).fillna(0).values, rows['TimeStamp'].map(SC[p][b]).fillna(0).values
        t = paired_bootstrap(rows['PassOrFail'].values, sa, sb, cluster=rows['TimeStamp'].values)
        BOOT.append(t.assign(제품=p, 틀='① 공식2차·행', A=a, B=b))
        # ② 샷 단위
        sa, sb = f['TimeStamp'].map(SC[p][a]).fillna(0).values, f['TimeStamp'].map(SC[p][b]).fillna(0).values
        t = paired_bootstrap(f['PassOrFail'].values, sa, sb)
        BOOT.append(t.assign(제품=p, 틀='② 전체·샷', A=a, B=b))
BOOT = pd.concat(BOOT, ignore_index=True)[['제품', '틀', 'A', 'B', '지표', '관측 차이(A−B)', 'CI 하한', 'CI 상한', 'P(A>B)', '판정']]
BOOT.to_csv(f'{OUT}/bootstrap_diff.csv', index=False, encoding='utf-8-sig')
display(BOOT.round(3))

,제품,틀,A,B,지표,관측 차이(A−B),CI 하한,CI 상한,P(A>B),판정
0,cn7,① 공식2차·행,C_maha+phys300,M_maha_free,ΔAP,-0.179,-0.263,0.012,0.127,차이 없음
1,cn7,① 공식2차·행,C_maha+phys300,M_maha_free,Δ상위10%,-0.059,-0.188,0.067,0.074,차이 없음
2,cn7,② 전체·샷,C_maha+phys300,M_maha_free,ΔAP,-0.120,-0.195,0.005,0.070,차이 없음
3,cn7,② 전체·샷,C_maha+phys300,M_maha_free,Δ상위10%,0.000,-0.067,0.000,0.000,차이 없음
4,cn7,① 공식2차·행,C_maha+phys300,T_phys300,ΔAP,0.044,-0.003,0.183,0.914,차이 없음
5,cn7,① 공식2차·행,C_maha+phys300,T_phys300,Δ상위10%,0.000,-0.125,0.067,0.065,차이 없음
6,cn7,② 전체·샷,C_maha+phys300,T_phys300,ΔAP,0.018,-0.002,0.084,0.912,차이 없음
7,cn7,② 전체·샷,C_maha+phys300,T_phys300,Δ상위10%,0.000,-0.133,0.200,0.421,차이 없음
8,cn7,① 공식2차·행,M_maha_free,T_past10,ΔAP,0.168,-0.042,0.359,0.889,차이 없음
9,cn7,① 공식2차·행,M_maha_free,T_past10,Δ상위10%,-0.118,-0.375,0.111,0.078,차이 없음


## C. 0 모드 제거 reference로 13·15번 재계산
기존 CN7 reference의 약 2/3가 0 모드였음 → 이전 CN7 비지도·앙상블 수치는 다시 계산한 값으로 교체

In [6]:
from run_compare import load_product
from unsup_eval import run_unsup, threshold_table
from model_phys import run_phys_models
from model_v2 import evaluate_v2, f1_v2, mask_transient

FN = {p: add_phys(load_product(p)[0]) for p in PRODUCTS}
# 15번 (순수 비지도)
OU = run_unsup(FN, REFNZ, baselines=False)
SU = evaluate_v2(OU); SU.to_csv(f'{OUT}/rerun15_summary.csv', index=False, encoding='utf-8-sig')
TU = threshold_table(OU); TU.to_csv(f'{OUT}/rerun15_threshold.csv', index=False, encoding='utf-8-sig')
c = ['product', 'method', 'AP', 'AP_lo', 'AP_hi', 'AP/base', 'hit@10%', 'AP_가스', 'hit10_가스', 'AP_미성형', 'hit10_미성형']
display(SU[c].round(3))
display(TU[TU['method'].isin(['maha|free', 'knn|free', 'iforest|free'])][
    ['product', 'method', 'q', '판정 불량(labeled)', 'labeled 판정 비율', 'TP', 'FP', 'Precision', 'Recall', 'F1']].round(3))

[cn7] 샷 1974, 불량 15
[rg3] 샷 627, 불량 32
  cn7 iforest|raw
  cn7 maha|raw
  cn7 knn|raw
  cn7 lof|raw
  cn7 pca|raw
  cn7 iforest|free
  cn7 maha|free
  cn7 knn|free
  cn7 lof|free
  cn7 pca|free
  cn7 iforest|phys(참고)
  cn7 maha|phys(참고)
  cn7 knn|phys(참고)
  cn7 lof|phys(참고)
  cn7 pca|phys(참고)
  rg3 iforest|raw
  rg3 maha|raw
  rg3 knn|raw
  rg3 lof|raw
  rg3 pca|raw
  rg3 iforest|free
  rg3 maha|free
  rg3 knn|free
  rg3 lof|free
  rg3 pca|free
  rg3 iforest|phys(참고)
  rg3 maha|phys(참고)
  rg3 knn|phys(참고)
  rg3 lof|phys(참고)
  rg3 pca|phys(참고)


,product,method,AP,AP_lo,AP_hi,AP/base,hit@10%,AP_가스,hit10_가스,AP_미성형,hit10_미성형
9,cn7,pca|free,0.214,0.013,0.420,28.149,4/15,0.009,1/11,0.751,3/4
6,cn7,maha|free,0.212,0.011,0.416,27.843,4/15,0.007,1/11,0.751,3/4
14,cn7,pca|phys(참고),0.210,0.009,0.412,27.663,3/15,0.007,0/11,0.751,3/4
11,cn7,maha|phys(참고),0.170,0.009,0.388,22.412,4/15,0.006,1/11,0.605,3/4
7,cn7,knn|free,0.129,0.010,0.354,17.027,4/15,0.007,1/11,0.442,3/4
12,cn7,knn|phys(참고),0.105,0.009,0.341,13.770,3/15,0.006,0/11,0.359,3/4
1,cn7,maha|raw,0.093,0.005,0.330,12.293,4/15,0.003,1/11,0.335,3/4
8,cn7,lof|free,0.092,0.010,0.321,12.152,3/15,0.007,0/11,0.303,3/4
5,cn7,iforest|free,0.081,0.008,0.299,10.628,4/15,0.006,1/11,0.270,3/4
10,cn7,iforest|phys(참고),0.070,0.009,0.272,9.202,4/15,0.006,1/11,0.230,3/4


,product,method,q,판정 불량(labeled),labeled 판정 비율,TP,FP,Precision,Recall,F1
25,cn7,iforest|free,0.950,89,0.045,3,86,0.034,0.200,0.058
26,cn7,iforest|free,0.980,45,0.023,3,42,0.067,0.200,0.100
27,cn7,iforest|free,0.990,23,0.012,3,20,0.130,0.200,0.158
28,cn7,iforest|free,0.995,18,0.009,3,15,0.167,0.200,0.182
29,cn7,iforest|free,0.999,4,0.002,1,3,0.250,0.067,0.105
30,cn7,maha|free,0.950,105,0.053,4,101,0.038,0.267,0.067
31,cn7,maha|free,0.980,45,0.023,3,42,0.067,0.200,0.100
32,cn7,maha|free,0.990,29,0.015,3,26,0.103,0.200,0.136
33,cn7,maha|free,0.995,19,0.010,3,16,0.158,0.200,0.176
34,cn7,maha|free,0.999,8,0.004,3,5,0.375,0.200,0.261


In [7]:
# 13번 (물리 기반 모델, 지도학습 포함 — 라벨 사용)
OP, _, _ = run_phys_models(FN, REFNZ)
OP = pd.concat([OP, mask_transient(OP[OP['method'].str.startswith('E')])], ignore_index=True)
SP = evaluate_v2(OP); SP.to_csv(f'{OUT}/rerun13_summary.csv', index=False, encoding='utf-8-sig')
display(SP[SP['method'].isin(['E2_free', 'E2_phys', 'U_rule|phys', 'E_phys+rule', 'E2_phys|과도제외', 'U_maha|free', 'U_knn|free'])][c].round(3))

  cn7 U_iforest|free 완료
  cn7 U_maha|free 완료
  cn7 U_knn|free 완료
  cn7 U_iforest|phys 완료
  cn7 U_maha|phys 완료
  cn7 U_knn|phys 완료
  cn7 U_iforest|phys_core 완료
  cn7 U_maha|phys_core 완료
  cn7 U_knn|phys_core 완료
  rg3 U_iforest|free 완료
  rg3 U_maha|free 완료
  rg3 U_knn|free 완료
  rg3 U_iforest|phys 완료
  rg3 U_maha|phys 완료
  rg3 U_knn|phys 완료
  rg3 U_iforest|phys_core 완료
  rg3 U_maha|phys_core 완료
  rg3 U_knn|phys_core 완료
  pooled LORO cn7_R1 완료 (학습 불량 33)
  pooled LORO cn7_R2 완료 (학습 불량 47)
  pooled LORO cn7_R3 완료 (학습 불량 47)
  pooled LORO cn7_R4 완료 (학습 불량 47)
  pooled LORO cn7_R5 완료 (학습 불량 47)
  pooled LORO cn7_R6 완료 (학습 불량 46)
  pooled LORO rg3_R1 완료 (학습 불량 47)
  pooled LORO rg3_R2 완료 (학습 불량 42)
  pooled LORO rg3_R3 완료 (학습 불량 37)
  pooled LORO rg3_R4 완료 (학습 불량 43)
  pooled LORO rg3_R5 완료 (학습 불량 41)
  pooled LORO rg3_R6 완료 (학습 불량 40)


,product,method,AP,AP_lo,AP_hi,AP/base,hit@10%,AP_가스,hit10_가스,AP_미성형,hit10_미성형
28,cn7,E2_free,0.213,0.012,0.419,27.968,4/15,0.008,1/11,0.751,3/4
1,cn7,U_maha|free,0.212,0.011,0.416,27.843,4/15,0.007,1/11,0.751,3/4
36,cn7,E2_phys|과도제외,0.139,0.011,0.392,18.237,4/15,0.007,1/11,0.480,3/4
2,cn7,U_knn|free,0.129,0.010,0.354,17.027,4/15,0.007,1/11,0.442,3/4
30,cn7,E2_phys,0.106,0.011,0.342,13.947,4/15,0.007,1/11,0.359,3/4
9,cn7,U_rule|phys,0.091,0.012,0.311,11.993,5/15,0.009,2/11,0.288,3/4
32,cn7,E_phys+rule,0.091,0.012,0.313,11.983,5/15,0.009,2/11,0.288,3/4
33,rg3,E_phys+rule,0.102,0.055,0.200,1.998,5/32,0.103,3/22,0.022,2/10
19,rg3,U_rule|phys,0.085,0.057,0.182,1.663,4/32,0.072,3/22,0.027,1/10
37,rg3,E2_phys|과도제외,0.066,0.047,0.126,1.296,5/32,0.058,4/22,0.017,1/10


## 해석 가이드
- **A**: 두 틀 모두 상위 = 최종 후보. 합산(C_*)이 두 구성 요소보다 둘 다 나으면 합산 채택
- **B**: CI가 0을 안 걸치면 '우세', 걸치면 '차이 없음' — 보고서 표현은 이 판정 그대로
- **C**: reference 적합도(labeled 판정 비율 ≈ 1−q)가 0 모드 제거 후 좋아졌는지, 이전 수치와 얼마나 바뀌었는지
- 미래 정보: T_global 제거, 합산은 reference 분포 기준이라 테스트 샷 간 순위를 쓰지 않음